In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, inspect
from pathlib import Path
import pandas as pd

HOME = Path.home()
CLIPS_DIR    = KWF_TEST_DIR / "clips"
OUTPUTS_DIR  = KWF_TEST_DIR / "outputs"
SCRIPTS_DIR  = PIPELINE_DIR / "scripts"
MODELS_DIR   = PIPELINE_DIR / "models"

# Same path setup as run_pipeline.py
sys.path.insert(0, str(SCRIPTS_DIR))

from aed_inference import score_folder

print("Signature:", inspect.signature(score_folder))
print("Model file exists:", (MODELS_DIR / "tinycnn_v3.pth").exists())

# The exact keyword arguments run_pipeline.py uses in step2_filter
master_kwargs = {"folder_path", "model_path", "output_csv"}
actual_params = set(inspect.signature(score_folder).parameters)
print("\nArguments run_pipeline.py passes that the function does NOT accept:", master_kwargs - actual_params or "none")

Signature: (folder_path, model_path, output_csv, recursive=True, audio_extensions=('.wav', '.flac', '.mp3', '.ogg'), batch_size=64, threshold=0.5, device=None)
Model file exists: True

Arguments run_pipeline.py passes that the function does NOT accept: none


In [2]:
import time

cnn_csv = OUTPUTS_DIR / "nb02_tinycnn_predictions.csv"

t0 = time.time()
score_folder(
    folder_path=str(CLIPS_DIR),
    model_path=str(MODELS_DIR / "tinycnn_v3.pth"),
    output_csv=str(cnn_csv),
)
print(f"\nRuntime: {time.time() - t0:.1f}s")

# Same filter logic as step2_filter in run_pipeline.py
df = pd.read_csv(cnn_csv)
meaningful = df[df["predicted_label"] == "meaningful"]["clip_name"].tolist()
print(f"Meaningful clips: {len(meaningful):,} / {len(df):,} total")

# Checks
print("\nColumns:", df.columns.tolist())
print("Rows:", len(df), "| Unique clip_names:", df["clip_name"].nunique())
print("Nulls per column:\n", df.isna().sum())
print("\nLabel counts:\n", df["predicted_label"].value_counts())

# Compare against the earlier command-line run (should be identical)
old = pd.read_csv(OUTPUTS_DIR / "tinycnn_predictions.csv").set_index("clip_name")["not_meaningful_prob"]
new = df.set_index("clip_name")["not_meaningful_prob"]
print("\nMax difference vs. CLI run:", (old - new.reindex(old.index)).abs().max())

Using model: tinycnn_v3.pth
  Version/notes:         TinyCNN v3 — 11213 not_meaningful clips (acoustic scan + model_inference_v1 + model_inference_v2)
  Device:                mps
  Val accuracy:          0.9984
  Not_meaningful recall: 1.000
  Not_meaningful F1:     0.990

Number of clips found: 1200

Scoring 1,200 clips on mps:0...
Batches: 19 x 64 clips



100%|██████████| 19/19 [00:08<00:00,  2.14it/s]


Saved 1,200 predictions to /Users/huyguy/kwf_test/outputs/nb02_tinycnn_predictions.csv
Skipped 0 missing/corrupt clips.

Runtime: 9.0s
Meaningful clips: 1,188 / 1,200 total

Columns: ['clip_name', 'audio_path', 'not_meaningful_prob', 'predicted_label', 'threshold', 'model_version']
Rows: 1200 | Unique clip_names: 1200
Nulls per column:
 clip_name              0
audio_path             0
not_meaningful_prob    0
predicted_label        0
threshold              0
model_version          0
dtype: int64

Label counts:
 predicted_label
meaningful        1188
not_meaningful      12
Name: count, dtype: int64

Max difference vs. CLI run: 0.0


In [3]:
flagged = df[df["predicted_label"] == "not_meaningful"].copy()
flagged["timestamp"] = pd.to_datetime(
    flagged["clip_name"].str.extract(r"(\d{8}_\d{6})")[0], format="%Y%m%d_%H%M%S"
)
print(flagged.sort_values("timestamp")[["clip_name", "timestamp", "not_meaningful_prob"]].to_string(index=False))

# Probabilities that are neither near 0 nor near 1
uncertain = df[(df["not_meaningful_prob"] > 0.05) & (df["not_meaningful_prob"] < 0.95)]
print(f"\nClips with probability between 0.05 and 0.95: {len(uncertain)}")

                       clip_name           timestamp  not_meaningful_prob
Audio_Moth_1_20250317_093130.wav 2025-03-17 09:31:30             0.560496
Audio_Moth_1_20250317_093639.wav 2025-03-17 09:36:39             0.889154
Audio_Moth_1_20250317_093824.wav 2025-03-17 09:38:24             0.997475
Audio_Moth_1_20250317_094151.wav 2025-03-17 09:41:51             1.000000
Audio_Moth_1_20250317_094203.wav 2025-03-17 09:42:03             0.975640
Audio_Moth_1_20250317_094630.wav 2025-03-17 09:46:30             0.866099
Audio_Moth_1_20250317_094657.wav 2025-03-17 09:46:57             1.000000
Audio_Moth_1_20250317_094736.wav 2025-03-17 09:47:36             0.999967
Audio_Moth_1_20250317_094739.wav 2025-03-17 09:47:39             0.992597
Audio_Moth_1_20250317_094742.wav 2025-03-17 09:47:42             0.999993
Audio_Moth_1_20250317_094745.wav 2025-03-17 09:47:45             0.999999
Audio_Moth_1_20250317_094751.wav 2025-03-17 09:47:51             0.997948

Clips with probability between 0.05 a

A quick hearing check shows that the majority of these clips are meaningful with clear footstep sound or human voice.

Step 2 on recorder 5: run Tiny CNN on the 1,747 split clips. Expect 1,747 rows, no skipped clips, the same columns as before, and clip names matching the Step 1 output.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, time, inspect
from pathlib import Path
import pandas as pd

HOME = Path.home()
SPLIT_OUT   = KWF_TEST_DIR / "split_clips"
OUTPUTS_DIR = KWF_TEST_DIR / "outputs"
MODELS_DIR = PIPELINE_DIR / "models"
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

import aed_inference
from aed_inference import score_folder

# Proof of which version is loaded
print("Loaded from:", aed_inference.__file__)
print("\n".join(l for l in inspect.getsource(aed_inference).splitlines() if "sys.path" in l or "architecture" in l))

csv5b = OUTPUTS_DIR / "nb02_recorder5_rerun_predictions.csv"
score_folder(
    folder_path=str(SPLIT_OUT),
    model_path=str(MODELS_DIR / "tinycnn_v3.pth"),
    output_csv=str(csv5b),
)

a = pd.read_csv(OUTPUTS_DIR / "nb02_recorder5_predictions.csv").set_index("clip_name")
b = pd.read_csv(csv5b).set_index("clip_name")
print("\nSame clips:", set(a.index) == set(b.index))
print("Max probability difference vs first run:", (a["not_meaningful_prob"] - b["not_meaningful_prob"].reindex(a.index)).abs().max())
print("Labels:", b["predicted_label"].value_counts().to_dict())

Loaded from: /Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline/scripts/aed_inference.py
# architecture.py sits next to this script in inference_pipeline/scripts/
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from architecture import TinyCNN
Using model: tinycnn_v3.pth
  Version/notes:         TinyCNN v3 — 11213 not_meaningful clips (acoustic scan + model_inference_v1 + model_inference_v2)
  Device:                mps
  Val accuracy:          0.9984
  Not_meaningful recall: 1.000
  Not_meaningful F1:     0.990

Number of clips found: 1747

Scoring 1,747 clips on mps:0...
Batches: 28 x 64 clips



100%|██████████| 28/28 [00:14<00:00,  1.94it/s]


Saved 1,747 predictions to /Users/huyguy/kwf_test/outputs/nb02_recorder5_rerun_predictions.csv
Skipped 0 missing/corrupt clips.

Same clips: True
Max probability difference vs first run: 0.0
Labels: {'meaningful': 1747}


Control: re-run the 1,200 recorder 1 clips with the team’s updated aed_inference.py. The pre-pull run flagged 12 clips. This shows whether “0 flagged” on recorder 5 is a real result.

In [2]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
CLIPS_DIR = KWF_TEST_DIR / "clips"
ctrl_csv = OUTPUTS_DIR / "nb02_rerun_postpull_predictions.csv"

score_folder(
    folder_path=str(CLIPS_DIR),
    model_path=str(MODELS_DIR / "tinycnn_v3.pth"),
    output_csv=str(ctrl_csv),
)

old = pd.read_csv(OUTPUTS_DIR / "nb02_tinycnn_predictions.csv").set_index("clip_name")
new = pd.read_csv(ctrl_csv).set_index("clip_name")

print("\nSame clips:", set(old.index) == set(new.index))
print("Max probability difference:", (old["not_meaningful_prob"] - new["not_meaningful_prob"].reindex(old.index)).abs().max())
print("Labels identical:", (old["predicted_label"] == new["predicted_label"].reindex(old.index)).all())
print("Label counts:", new["predicted_label"].value_counts().to_dict())

Using model: tinycnn_v3.pth
  Version/notes:         TinyCNN v3 — 11213 not_meaningful clips (acoustic scan + model_inference_v1 + model_inference_v2)
  Device:                mps
  Val accuracy:          0.9984
  Not_meaningful recall: 1.000
  Not_meaningful F1:     0.990

Number of clips found: 1200

Scoring 1,200 clips on mps:0...
Batches: 19 x 64 clips



100%|██████████| 19/19 [00:08<00:00,  2.22it/s]


Saved 1,200 predictions to /Users/huyguy/kwf_test/outputs/nb02_rerun_postpull_predictions.csv
Skipped 0 missing/corrupt clips.

Same clips: True
Max probability difference: 0.0
Labels identical: True
Label counts: {'meaningful': 1188, 'not_meaningful': 12}


## Observations (Step 2)
- score_folder(folder_path, model_path, output_csv) matches the call in run_pipeline.py step2_filter.
- Pre-pull run (our one-line import fix) and post-pull run (team's version) are identical on recorder 1: max probability difference 0.0, same 12 of 1,200 clips flagged.
- Recorder 5 (1,747 clips from Step 1): 0 skipped, clip_names match the Step 1 output exactly, all meaningful (max probability 0.0).
- Consequence for Step 7: the meaningful filter would drop 12 clips from recorder 1 and 0 from recorder 5.
- In a manual listen, 9 of the 12 flagged recorder 1 clips sounded meaningful.
- Output rows are not in chronological order (Step 7 features such as Volume_Spike_15s need time order, so sorting must happen downstream).